# Systematic Review Screening Pipeline
## AI and Machine Learning for Bacterial Colony Identification on Agar Plate Images
*Modeled after the clinical medicine systematic review screening workflow (`clinical_medicine_screening.ipynb`)*

---

### Workflow Overview:
1. **Load Screening Instructions & Deduplicated Corpus**: Load the comprehensive criteria from `bacteria_screening_instructions.txt` and the 16,013 deduplicated papers from `deduplicated_papers.csv`.
2. **Partition into 5 Groups**: The 16,013 papers are split into **5 balanced groups** (~3,203 papers each) for staged processing and manual human auditing.
3. **Screen Group 1 First**:
   - Option A: **OpenAI Batch API** (50% cheaper, asynchronous, standard for large systematic reviews).
   - Option B: **Direct API Screening** (with persistent progress checkpointing, live `tqdm` progress, and exponential backoff retry).
4. **Groups 2 to 5 Staged (Protected)**: Code is written for all groups, but **locked** (`RUN_GROUP_X = False`) so they will not run until Group 1 is audited and approved.
5. **Parse, Audit & Merge Results**: Extract JSON decisions (`Include?`, `Exclusion_reason`, `Uncertain`, `Comments`), merge with metadata, and compute agreement metrics (Cohen's Kappa, Sensitivity/Specificity).


## 1. Setup, Environment & Configuration


In [47]:
import os
import sys
import json
import csv
import copy
import time
import getpass
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

# Attempt imports for optional data science libraries
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

# Create project directory structure
os.makedirs("screening_groups", exist_ok=True)
os.makedirs("batch_requests", exist_ok=True)
os.makedirs("screening_results", exist_ok=True)

# API Configuration
# Read from environment variable or set your OpenAI API key directly below:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    # Enter your OpenAI API key below if not set in environment:
    OPENAI_API_KEY = ""  # <-- Paste 'sk-...' here if not using env var

# Screening model selection
# Options: 'gpt-4o-mini' (recommended: fast, highly accurate, low cost), 'gpt-4o', 'o3-mini'
DEFAULT_MODEL = "gpt-4o-mini"

print(f"Setup complete.")
print(f"Pandas available: {pd is not None}")
print(f"OpenAI client available: {OpenAI is not None}")
print(f"OpenAI API Key configured: {bool(OPENAI_API_KEY)}")
print(f"Default screening model: {DEFAULT_MODEL}")


Setup complete.
Pandas available: True
OpenAI client available: True
Default screening model: gpt-4o-mini


## 2. Load Screening Instructions & Deduplicated Papers


In [48]:
# 1. Load screening instructions
INSTRUCTIONS_FILE = "bacteria_screening_instructions.txt"
with open(INSTRUCTIONS_FILE, "r", encoding="utf-8") as f:
    INSTRUCTIONS_RAW = f.read()

# Verify that instructions are valid JSON
try:
    instructions_json = json.loads(INSTRUCTIONS_RAW)
    print("Screening instructions loaded successfully as valid JSON.")
    print("Topic:", instructions_json["instructions"].get("review_topic"))
    print("Period:", instructions_json["instructions"].get("review_period"))
except Exception as e:
    print("Loaded instructions as raw text:", str(e))

# 2. Load deduplicated papers
PAPERS_FILE = "deduplicated_papers.csv"
all_papers = []
with open(PAPERS_FILE, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    all_papers = list(reader)

total_papers = len(all_papers)
print(f"\nTotal deduplicated papers: {total_papers:,}")

# Database breakdown
db_counts = Counter(p.get("Primary_Database", "Unknown") for p in all_papers)
print("\nDatabase Breakdown:")
for db, cnt in db_counts.most_common():
    print(f"  - {db:15}: {cnt:6,} ({cnt/total_papers*100:.1f}%)")


Screening instructions loaded successfully as valid JSON.
Topic: Artificial Intelligence and Machine Learning for Bacterial Colony Identification on Agar Plate Images
Period: {'start_year': 2010, 'end_year': 2026, 'inclusive': True}

Total deduplicated papers: 16,013

Database Breakdown:
  - Scopus         : 14,941 (93.3%)
  - PubMed         :    709 (4.4%)
  - SpringerLink   :    241 (1.5%)
  - IEEE Xplore    :     98 (0.6%)
  - Embase         :     24 (0.1%)


## 3. Divide Dataset into 5 Groups (For Manual Checking & Staged Screening)
The 16,013 papers are divided into **5 balanced groups**:
- **Group 1**: 3,203 papers (Index 0 – 3,202)
- **Group 2**: 3,203 papers (Index 3,203 – 6,405)
- **Group 3**: 3,203 papers (Index 6,406 – 9,608)
- **Group 4**: 3,202 papers (Index 9,609 – 12,810)
- **Group 5**: 3,202 papers (Index 12,811 – 16,012)

Each group is saved with dedicated columns for manual human verification (`Human_Include`, `Human_Exclusion_Reason`, `Human_Comments`).


In [49]:
n_groups = 5
base_size = total_papers // n_groups
remainder = total_papers % n_groups

group_partitions = {}
start_idx = 0

fieldnames = [
    "Study_ID", "Group", "Group_Index", "Global_Index",
    "Primary_Database", "All_Databases", "Primary_ID", "All_IDs",
    "Title", "Authors", "Year", "Journal", "DOI", "Abstract",
    "Human_Include", "Human_Exclusion_Reason", "Human_Comments",
    "LLM_Include", "LLM_Exclusion_Reason", "LLM_Uncertain", "LLM_Comments"
]

print("Generating 5 Group Partitions...")
for g in range(n_groups):
    size = base_size + (1 if g < remainder else 0)
    end_idx = start_idx + size
    group_num = g + 1
    group_papers = all_papers[start_idx:end_idx]
    
    csv_path = f"screening_groups/group_{group_num}_papers.csv"
    jsonl_path = f"screening_groups/group_{group_num}_papers.jsonl"
    
    with open(csv_path, "w", encoding="utf-8-sig", newline="") as f_csv, \
         open(jsonl_path, "w", encoding="utf-8") as f_jsonl:
        
        writer = csv.DictWriter(f_csv, fieldnames=fieldnames)
        writer.writeheader()
        
        processed_group_records = []
        for g_idx, p in enumerate(group_papers):
            study_id = f"G{group_num}_{g_idx+1:04d}"
            record = {
                "Study_ID": study_id,
                "Group": f"Group {group_num}",
                "Group_Index": g_idx + 1,
                "Global_Index": start_idx + g_idx,
                "Primary_Database": p.get("Primary_Database", ""),
                "All_Databases": p.get("All_Databases", ""),
                "Primary_ID": p.get("Primary_ID", ""),
                "All_IDs": p.get("All_IDs", ""),
                "Title": p.get("Title", "").strip(),
                "Authors": p.get("Authors", "").strip(),
                "Year": p.get("Year", "").strip(),
                "Journal": p.get("Journal", "").strip(),
                "DOI": p.get("DOI", "").strip(),
                "Abstract": p.get("Abstract", "").strip(),
                "Human_Include": "",
                "Human_Exclusion_Reason": "",
                "Human_Comments": "",
                "LLM_Include": "",
                "LLM_Exclusion_Reason": "",
                "LLM_Uncertain": "",
                "LLM_Comments": ""
            }
            writer.writerow(record)
            f_jsonl.write(json.dumps(record, ensure_ascii=False) + "\n")
            processed_group_records.append(record)
            
    group_partitions[group_num] = {
        "count": len(group_papers),
        "start_idx": start_idx,
        "end_idx": end_idx - 1,
        "csv_path": csv_path,
        "jsonl_path": jsonl_path,
        "records": processed_group_records
    }
    
    print(f"  Group {group_num}: {len(group_papers):,} papers (Global {start_idx:,} to {end_idx-1:,}) -> {csv_path}")
    start_idx = end_idx


Generating 5 Group Partitions...
  Group 1: 3,203 papers (Global 0 to 3,202) -> screening_groups/group_1_papers.csv
  Group 2: 3,203 papers (Global 3,203 to 6,405) -> screening_groups/group_2_papers.csv
  Group 3: 3,203 papers (Global 6,406 to 9,608) -> screening_groups/group_3_papers.csv
  Group 4: 3,202 papers (Global 9,609 to 12,810) -> screening_groups/group_4_papers.csv
  Group 5: 3,202 papers (Global 12,811 to 16,012) -> screening_groups/group_5_papers.csv


## 4. Helper Functions: Batch Generation & Direct API Screener


In [50]:
def create_batch_requests_file(group_records, output_file, model="gpt-4o-mini"):
    """
    Builds an OpenAI Batch API compatible JSONL file.
    Uses /v1/chat/completions with JSON response format.
    """
    batch_lines = []
    for r in group_records:
        title = r["Title"]
        abstract = r["Abstract"] if r["Abstract"] else "[No abstract provided in bibliographic record]"
        prompt_content = f"Title: {title}\nAbstract: {abstract}"
        
        request_entry = {
            "custom_id": r["Study_ID"],
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "temperature": 0.0,
                "response_format": {"type": "json_object"},
                "messages": [
                    {"role": "system", "content": INSTRUCTIONS_RAW},
                    {"role": "user", "content": prompt_content}
                ]
            }
        }
        batch_lines.append(request_entry)
        
    with open(output_file, "w", encoding="utf-8") as f:
        for entry in batch_lines:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
            
    print(f"Generated {len(batch_lines):,} batch requests -> {output_file}")
    return output_file

def screen_single_study_direct(client, title, abstract, model="gpt-4o-mini", max_retries=3):
    """
    Screens a single study directly via OpenAI Chat Completions API with exponential backoff.
    """
    if not abstract:
        abstract = "[No abstract provided in bibliographic record]"
    prompt_content = f"Title: {title}\nAbstract: {abstract}"
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0.0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": INSTRUCTIONS_RAW},
                    {"role": "user", "content": prompt_content}
                ],
                timeout=30.0
            )
            raw_text = response.choices[0].message.content
            parsed = json.loads(raw_text)
            return {
                "Include?": parsed.get("Include?", "uncertain"),
                "Exclusion_reason": parsed.get("Exclusion_reason"),
                "Uncertain": parsed.get("Uncertain", "no"),
                "Comments": parsed.get("Comments", "")
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {
                    "Include?": "error",
                    "Exclusion_reason": "API_ERROR",
                    "Uncertain": "yes",
                    "Comments": f"API request error: {str(e)}"
                }


## 5. GROUP 1 SCREENING (Execute First)
> "Take the first group of papers to do first"

Prepare and execute the screening for **Group 1** (3,203 papers).

You have two ways to execute screening:
- **Option A (Batch API - Recommended for full 3,203 papers)**: 50% discount, zero timeout/connection drop risk.
- **Option B (Direct API - Recommended for Pilot testing)**: Real-time progress with continuous auto-checkpointing (test 5–10 papers first).


In [ ]:
# 5.1 Generate Batch Request File for Group 1 (3,203 papers)
group1_records = group_partitions[1]["records"]
batch_group1_file = "batch_requests/bacteria_screening_batch_group_1.jsonl"

create_batch_requests_file(group1_records, batch_group1_file, model=DEFAULT_MODEL)


### Option A: OpenAI Batch API (Recommended for Full 3,203 Papers)
This uploads `bacteria_screening_batch_group_1.jsonl` to OpenAI Batch API (50% cheaper, asynchronous).
Once submitted, OpenAI processes the papers in parallel. You can monitor the status and download results using the checker below.


In [ ]:
def submit_group_batch(group_num=1):
    """
    Uploads batch request file and submits batch job to OpenAI.
    Saves the batch ID to batch_requests/group_{group_num}_batch_info.json.
    """
    if not OPENAI_API_KEY:
        print("ERROR: OPENAI_API_KEY is not configured.")
        print("Please set OPENAI_API_KEY in Section 1 (Cell 2) and re-run that cell.")
        return None
        
    client = OpenAI(api_key=OPENAI_API_KEY)
    batch_file_path = f"batch_requests/bacteria_screening_batch_group_{group_num}.jsonl"
    
    if not os.path.exists(batch_file_path):
        print(f"Error: {batch_file_path} not found. Please run cell 5.1 above first.")
        return None
        
    print(f"1. Uploading {batch_file_path} to OpenAI...")
    with open(batch_file_path, "rb") as f:
        batch_file = client.files.create(file=f, purpose="batch")
    print(f"   ✓ Uploaded successfully (File ID: {batch_file.id})")
    
    print("2. Creating Batch Job on OpenAI...")
    batch_job = client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"group": f"group_{group_num}", "project": "bacteria_screening"}
    )
    print(f"   ✓ Batch submitted successfully!")
    print(f"     Batch ID : {batch_job.id}")
    print(f"     Status   : {batch_job.status}")
    
    info_path = f"batch_requests/group_{group_num}_batch_info.json"
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump({
            "batch_id": batch_job.id,
            "input_file_id": batch_file.id,
            "created_at": batch_job.created_at,
            "group_num": group_num
        }, f, indent=2)
    print(f"   ✓ Batch info saved to -> {info_path}")
    print("\n--> Now run the cell below to monitor status or download results.")
    return batch_job.id

# Run this function to submit Group 1 Batch to OpenAI:
# group1_batch_id = submit_group_batch(group_num=1)


In [ ]:
def check_and_download_batch(group_num=1, batch_id=None):
    """
    Checks status of an active batch job. When completed, automatically downloads
    the output file to screening_results/group_{group_num}_batch_output.jsonl.
    """
    if not OPENAI_API_KEY:
        print("ERROR: OPENAI_API_KEY is not configured.")
        print("Please set OPENAI_API_KEY in Section 1 (Cell 2).")
        return None
        
    client = OpenAI(api_key=OPENAI_API_KEY)
    info_path = f"batch_requests/group_{group_num}_batch_info.json"
    
    if not batch_id:
        if os.path.exists(info_path):
            with open(info_path, "r", encoding="utf-8") as f:
                batch_id = json.load(f).get("batch_id")
        else:
            print(f"No batch info found for Group {group_num}. Run submit_group_batch first.")
            return None
            
    batch_job = client.batches.retrieve(batch_id)
    print(f"=== Group {group_num} Batch Status ===")
    print(f"  Batch ID : {batch_job.id}")
    print(f"  Status   : {batch_job.status.upper()}")
    
    counts = batch_job.request_counts
    if counts:
        print(f"  Progress : {counts.completed:,} completed / {counts.failed:,} failed / {counts.total:,} total")
        
    if batch_job.status == "completed":
        output_file_id = batch_job.output_file_id
        print(f"\n✓ Batch finished! Downloading output file ({output_file_id})...")
        content = client.files.content(output_file_id).read()
        output_path = f"screening_results/group_{group_num}_batch_output.jsonl"
        with open(output_path, "wb") as f:
            f.write(content)
        print(f"✓ Results saved to -> {output_path}")
        print("\n--> Now run 'process_and_merge_group_results(group_num=1)' below to merge decisions!")
    elif batch_job.status in ("in_progress", "validating", "finalizing"):
        print(f"\nBatch is currently {batch_job.status}. Check back in 15–30 minutes.")
    elif batch_job.status in ("failed", "expired", "cancelled"):
        print(f"\nWarning: Batch ended with status: {batch_job.status}.")
        if hasattr(batch_job, "errors") and batch_job.errors:
            print(f"Errors: {batch_job.errors}")
            
    return batch_job

# Run to check batch status and download results when ready:
# check_and_download_batch(group_num=1)


### Option B: Direct API Screening (Alternative: Pilot Test or Streaming)
Direct API screening makes live calls to `client.chat.completions.create` with automatic retry and checkpointing.
It saves progress after every paper to `screening_results/group_1_direct_checkpoint.jsonl`.

> **Tip**: Run a **5-paper pilot test** to verify that inclusion/exclusion decisions follow your rubric before launching the full run.


In [ ]:
def run_group_direct_screening(group_records, group_num=1, model=DEFAULT_MODEL, sample_limit=None):
    """
    Runs direct screening on a group of papers with continuous auto-saving and checkpointing.
    Set sample_limit (e.g., 5 or 20) for a pilot test, or None for the full group.
    """
    if OpenAI is None or not OPENAI_API_KEY:
        print("ERROR: OpenAI client or OPENAI_API_KEY is not configured.")
        print("Please set your OPENAI_API_KEY in Section 1 (Cell 2) and re-run that cell.")
        return None
        
    client = OpenAI(api_key=OPENAI_API_KEY)
    checkpoint_file = f"screening_results/group_{group_num}_direct_checkpoint.jsonl"
    
    # Load existing progress if any
    screened_data = {}
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                screened_data[item["Study_ID"]] = item
        print(f"Resuming from checkpoint: {len(screened_data):,} papers already screened.")
        
    target_records = group_records[:sample_limit] if sample_limit else group_records
    print(f"Target papers to screen in this run: {len(target_records):,} papers")
    
    with open(checkpoint_file, "a", encoding="utf-8") as f_out:
        for r in tqdm(target_records, desc=f"Screening Group {group_num}"):
            sid = r["Study_ID"]
            if sid in screened_data:
                continue
                
            decision = screen_single_study_direct(client, r["Title"], r["Abstract"], model=model)
            
            result_item = copy.deepcopy(r)
            inc_val = str(decision.get("Include?", "")).strip().lower()
            result_item["LLM_Include"] = inc_val
            result_item["LLM_Exclusion_Reason"] = decision.get("Exclusion_reason") or ""
            result_item["LLM_Uncertain"] = decision.get("Uncertain", "")
            result_item["LLM_Comments"] = decision.get("Comments", "")
            
            if inc_val == "yes":
                result_item["screening_decision"] = "Include"
            elif str(decision.get("Uncertain", "")).lower() == "yes" or inc_val == "uncertain":
                result_item["screening_decision"] = "Uncertain"
            else:
                result_item["screening_decision"] = "Exclude"
                
            result_item["exclusion_reasons"] = decision.get("Exclusion_reason") or ""
            result_item["screening_error"] = decision.get("Comments", "") if inc_val == "error" else ""
            
            f_out.write(json.dumps(result_item, ensure_ascii=False) + "\n")
            f_out.flush()
            screened_data[sid] = result_item
            
    print(f"\n✓ Screening complete: {len(screened_data):,} papers recorded in {checkpoint_file}")
    return list(screened_data.values())

# PILOT TEST: Run first 5 papers of Group 1 (Fast test to verify criteria):
# pilot_results = run_group_direct_screening(group1_records, group_num=1, sample_limit=5)

# FULL RUN: Screen all 3,203 papers directly via API (Uncomment when ready):
# full_results = run_group_direct_screening(group1_records, group_num=1, sample_limit=None)


### Parse, Merge & Export Group 1 Results
Once Group 1 screening is complete (either via **Batch API output** or **Direct API runner**), this merges the decisions with the original bibliographic records and exports:
1. `screening_results/group_1_screened.csv` (Full dataset with decisions)
2. `screening_results/group_1_included_studies.csv` (Studies included for next stage)


In [ ]:
def process_and_merge_group_results(group_num=1):
    """
    Merges LLM screening decisions with original group papers,
    exporting both full screened CSV and an included studies subset.
    """
    checkpoint_file = f"screening_results/group_{group_num}_direct_checkpoint.jsonl"
    batch_output_file = f"screening_results/group_{group_num}_batch_output.jsonl"
    
    decisions_by_id = {}
    
    # 1. Check if Batch API results exist
    if os.path.exists(batch_output_file):
        with open(batch_output_file, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                res = json.loads(line)
                sid = res.get("custom_id")
                try:
                    content = res["response"]["body"]["choices"][0]["message"]["content"]
                    parsed = json.loads(content)
                    decisions_by_id[sid] = parsed
                except Exception:
                    pass
        print(f"Loaded {len(decisions_by_id):,} decisions from Batch API output.")
        
    # 2. Check direct screening checkpoint
    elif os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                res = json.loads(line)
                sid = res.get("Study_ID")
                decisions_by_id[sid] = {
                    "Include?": res.get("LLM_Include") or ("yes" if res.get("screening_decision") == "Include" else "no"),
                    "Exclusion_reason": res.get("LLM_Exclusion_Reason") or res.get("exclusion_reasons"),
                    "Uncertain": res.get("LLM_Uncertain") or ("yes" if res.get("screening_decision") == "Uncertain" else "no"),
                    "Comments": res.get("LLM_Comments") or res.get("screening_error", "")
                }
        print(f"Loaded {len(decisions_by_id):,} decisions from Direct Screening checkpoint.")
    else:
        print(f"No screening output found for Group {group_num} yet.")
        print("Please run either Option A (Batch API) or Option B (Direct API) first.")
        return None
        
    # Merge with group papers
    group_csv = f"screening_groups/group_{group_num}_papers.csv"
    output_screened_csv = f"screening_results/group_{group_num}_screened.csv"
    output_included_csv = f"screening_results/group_{group_num}_included_studies.csv"
    
    with open(group_csv, "r", encoding="utf-8-sig") as f_in:
        rows = list(csv.DictReader(f_in))
        
    included_count = 0
    excluded_count = 0
    uncertain_count = 0
    
    # Build fieldnames ensuring all standard and analysis columns exist
    extended_fieldnames = list(rows[0].keys()) if rows else []
    for col in ["screening_decision", "exclusion_reasons", "screening_error", "LLM_Include", "LLM_Exclusion_Reason", "LLM_Uncertain", "LLM_Comments"]:
        if col not in extended_fieldnames:
            extended_fieldnames.append(col)
            
    included_rows = []
    for r in rows:
        sid = r["Study_ID"]
        if sid in decisions_by_id:
            d = decisions_by_id[sid]
            inc = str(d.get("Include?", "")).strip().lower()
            is_unc = str(d.get("Uncertain", "")).strip().lower() == "yes"
            excl_reason = str(d.get("Exclusion_reason", "") or "")
            comments = str(d.get("Comments", "") or "")
            
            r["LLM_Include"] = inc
            r["LLM_Exclusion_Reason"] = excl_reason
            r["LLM_Uncertain"] = "yes" if is_unc else "no"
            r["LLM_Comments"] = comments
            
            if inc == "yes":
                r["screening_decision"] = "Include"
                included_count += 1
                included_rows.append(r)
            elif is_unc or inc == "uncertain":
                r["screening_decision"] = "Uncertain"
                uncertain_count += 1
            else:
                r["screening_decision"] = "Exclude"
                excluded_count += 1
                
            r["exclusion_reasons"] = excl_reason
            r["screening_error"] = comments if inc == "error" else ""
        else:
            r["screening_decision"] = ""
            r["exclusion_reasons"] = ""
            r["screening_error"] = ""
            
    # Save full screened CSV
    with open(output_screened_csv, "w", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=extended_fieldnames)
        writer.writeheader()
        writer.writerows(rows)
        
    # Save included studies CSV
    with open(output_included_csv, "w", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=extended_fieldnames)
        writer.writeheader()
        writer.writerows(included_rows)
        
    print(f"\n=== Group {group_num} Screening Summary ===")
    print(f"Total Papers in Group : {len(rows):,}")
    print(f"Papers Screened       : {len(decisions_by_id):,}")
    print(f"Included (Proceeding) : {included_count:,} ({included_count/max(len(decisions_by_id),1)*100:.1f}%)")
    print(f"Excluded              : {excluded_count:,} ({excluded_count/max(len(decisions_by_id),1)*100:.1f}%)")
    print(f"Uncertain Flagged     : {uncertain_count:,} ({uncertain_count/max(len(decisions_by_id),1)*100:.1f}%)")
    print(f"Full Results Saved    : {output_screened_csv}")
    print(f"Included Studies Saved: {output_included_csv}")
    return rows

# Run merger if results exist:
process_and_merge_group_results(group_num=1)


These cells are prepared for Groups 2, 3, 4, and 5, but have been guarded with `RUN_GROUP_X = False`.


In [54]:
# GROUP 2 SCREENING - PENDING GROUP 1 EVALUATION & MANUAL AUDIT
RUN_GROUP_2 = False 

if not RUN_GROUP_2:
    print("Group 2 execution is LOCKED. (Set RUN_GROUP_2 = True when ready)")
else:
    print("Preparing Group 2 batch requests...")
    group2_records = group_partitions[2]["records"]
    batch_group2_file = "batch_requests/bacteria_screening_batch_group_2.jsonl"
    create_batch_requests_file(group2_records, batch_group2_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group2_records, group_num=2)


Group 2 execution is LOCKED. (Set RUN_GROUP_2 = True when ready)


In [55]:
# GROUP 3 SCREENING 
RUN_GROUP_3 = False 

if not RUN_GROUP_3:
    print("Group 3 execution is LOCKED. (Set RUN_GROUP_3 = True when ready)")
else:
    print("Preparing Group 3 batch requests...")
    group3_records = group_partitions[3]["records"]
    batch_group3_file = "batch_requests/bacteria_screening_batch_group_3.jsonl"
    create_batch_requests_file(group3_records, batch_group3_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group3_records, group_num=3)


Group 3 execution is LOCKED. (Set RUN_GROUP_3 = True when ready)


In [56]:
# GROUP 4 SCREENING
RUN_GROUP_4 = False 

if not RUN_GROUP_4:
    print("Group 4 execution is LOCKED. (Set RUN_GROUP_4 = True when ready)")
else:
    print("Preparing Group 4 batch requests...")
    group4_records = group_partitions[4]["records"]
    batch_group4_file = "batch_requests/bacteria_screening_batch_group_4.jsonl"
    create_batch_requests_file(group4_records, batch_group4_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group4_records, group_num=4)


Group 4 execution is LOCKED. (Set RUN_GROUP_4 = True when ready)


In [57]:
# GROUP 5 SCREENING 
RUN_GROUP_5 = False 

if not RUN_GROUP_5:
    print("Group 5 execution is LOCKED. (Set RUN_GROUP_5 = True when ready)")
else:
    print("Preparing Group 5 batch requests...")
    group5_records = group_partitions[5]["records"]
    batch_group5_file = "batch_requests/bacteria_screening_batch_group_5.jsonl"
    create_batch_requests_file(group5_records, batch_group5_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group5_records, group_num=5)


Group 5 execution is LOCKED. (Set RUN_GROUP_5 = True when ready)


In [58]:
RESULTS_DIR = Path("screening_results")
TOTAL_GROUPS = 5

print("=" * 65)
print("POST-SCREENING STATUS")
print("=" * 65)

group_stats = []
all_results = []
all_included = []

for g in range(1, TOTAL_GROUPS + 1):
    csv_path = RESULTS_DIR / f"group_{g}_screened.csv"
    jsonl_path = RESULTS_DIR / f"group_{g}_screened.jsonl"
    checkpoint_path = RESULTS_DIR / f"group_{g}_direct_checkpoint.jsonl"
    batch_out_path = RESULTS_DIR / f"group_{g}_batch_output.jsonl"
    
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        total = len(df)
        
        if "screening_decision" in df.columns:
            decisions = df["screening_decision"].value_counts().to_dict()
            included = decisions.get("Include", 0)
            excluded = decisions.get("Exclude", 0)
            uncertain = decisions.get("Uncertain", 0)
        elif "LLM_Include" in df.columns:
            included = (df["LLM_Include"].astype(str).str.lower() == "yes").sum()
            excluded = (df["LLM_Include"].astype(str).str.lower() == "no").sum()
            uncertain = ((df["LLM_Uncertain"].astype(str).str.lower() == "yes") | (df["LLM_Include"].astype(str).str.lower() == "uncertain")).sum()
        else:
            included, excluded, uncertain = 0, 0, 0
            
        errors = df["screening_error"].notna().sum() if "screening_error" in df.columns else 0
        
        # Check if full or partial pilot
        status_label = "COMPLETED" if (included + excluded + uncertain) >= total else f"PILOT ({included+excluded+uncertain}/{total})"
        
        group_stats.append({
            "Group": f"Group {g}",
            "Status": status_label,
            "Screened": included + excluded + uncertain,
            "Included": included,
            "Excluded": excluded,
            "Uncertain": uncertain,
            "Errors": errors
        })
        all_results.append(df)
        if "screening_decision" in df.columns:
            all_included.append(df[df["screening_decision"] == "Include"])
        elif "LLM_Include" in df.columns:
            all_included.append(df[df["LLM_Include"].astype(str).str.lower() == "yes"])
            
    elif jsonl_path.exists() or checkpoint_path.exists():
        # Checkpoint exists but not yet merged to CSV
        source_file = jsonl_path if jsonl_path.exists() else checkpoint_path
        rows = []
        with open(source_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        total = len(rows)
        group_stats.append({
            "Group": f"Group {g}",
            "Status": "IN PROGRESS (UNMERGED)",
            "Screened": total,
            "Included": 0,
            "Excluded": 0,
            "Uncertain": 0,
            "Errors": 0
        })
    else:
        group_stats.append({
            "Group": f"Group {g}",
            "Status": "NOT SCREENED",
            "Screened": 0,
            "Included": 0,
            "Excluded": 0,
            "Uncertain": 0,
            "Errors": 0
        })

# 1. Group-by-Group Table
stats_df = pd.DataFrame(group_stats)
print("\n--- Group Status ---")
print(stats_df.to_string(index=False))

# 2. Overall Totals
total_screened = stats_df["Screened"].sum()
total_included = stats_df["Included"].sum()
total_excluded = stats_df["Excluded"].sum()
total_uncertain = stats_df["Uncertain"].sum()

print("\n" + "=" * 65)
print(f"Total Papers Screened : {total_screened:,}")
print(f"Papers Left (INCLUDED): {total_included:,}  ({total_included/max(total_screened,1)*100:.1f}%)")
print(f"Papers EXCLUDED       : {total_excluded:,}  ({total_excluded/max(total_screened,1)*100:.1f}%)")
print(f"Papers UNCERTAIN      : {total_uncertain:,}  ({total_uncertain/max(total_screened,1)*100:.1f}%)")
print("=" * 65)

# 3. Top Exclusion Reasons (if any papers have been screened)
if all_results and total_excluded > 0:
    combined_df = pd.concat(all_results, ignore_index=True)
    reason_col = "exclusion_reasons" if "exclusion_reasons" in combined_df.columns else ("LLM_Exclusion_Reason" if "LLM_Exclusion_Reason" in combined_df.columns else None)
    if reason_col:
        is_excl = (combined_df["screening_decision"] == "Exclude") if "screening_decision" in combined_df.columns else (combined_df["LLM_Include"].astype(str).str.lower() == "no")
        excl_reasons = combined_df[is_excl][reason_col].dropna()
        reason_counts = Counter()
        for r in excl_reasons:
            for item in str(r).split(";"):
                item = item.strip()
                if item and item.lower() != "nan":
                    reason_counts[item] += 1
        
        if reason_counts:
            print("\n--- Top Exclusion Reasons ---")
            for reason, count in reason_counts.most_common(10):
                print(f"  {reason:<35}: {count:>5} ({count/total_excluded*100:.1f}%)")


POST-SCREENING STATUS

--- Group Status ---
  Group       Status  Screened  Included  Excluded  Uncertain  Errors
Group 1 NOT SCREENED         0         0         0          0       0
Group 2 NOT SCREENED         0         0         0          0       0
Group 3 NOT SCREENED         0         0         0          0       0
Group 4 NOT SCREENED         0         0         0          0       0
Group 5 NOT SCREENED         0         0         0          0       0

Total Papers Screened : 0
Papers Left (INCLUDED): 0  (0.0%)
Papers EXCLUDED       : 0  (0.0%)
Papers UNCERTAIN      : 0  (0.0%)
